**Question:** Can a gradient-boosted model beat the closed-form ELO baseline, with and without ELO as an input?

**Assumptions:** Chronological split only; calibration is fit on a later held-out slice of train, never on test.

**Method:** Train Exp 0 (ELO formula), Exp A (ELO-free XGBoost) and Exp B (XGBoost with ELO) with time-series CV on train.

**Result:** Test Brier: ELO 0.1771, A 0.1969, B 0.1670 (July 2026 snapshot).

# 03 — Model Training: Experiment 0 / A / B

Loads the engineered, tagged, time-split frame from `02_feature_engineering.ipynb` and trains the
three comparable experiments:

- **Experiment 0 — ELO baseline (no ML):** closed-form `1/(1+10^(-elo_diff/400))` applied directly
  to the TEST set. No fitting.
- **Experiment A — ELO-free XGBoost:** `experiment_a_cols` (27 cols, no `elo_diff`/`elo_confidence`
  raw ELO). ELO-substitute features (forma, career win-rate, H2H, razina, debut, clutch) carry the
  opponent-quality signal instead.
- **Experiment B — ELO-included XGBoost:** `experiment_b_cols` (28 cols = Exp A + `elo_diff`).

**Split:** uses the EXISTING time-based `Split` column from `features.parquet` (train: 2017-05-08 to
2025-01-18, 27,404 rows / 13,702 matches; test: 2025-01-19 to 2026-07-08, 6,852 rows / 3,426
matches). Never re-split randomly — both perspective rows of a match already share the same split.

**Calibration discipline:** hyperparameter sanity-checked via match-level `TimeSeriesSplit` CV
*within TRAIN only*. Final model is fit on a chronological "core" sub-slice of train (with a small
early-stopping eval tail), then wrapped in `CalibratedClassifierCV(cv='prefit')` fit on a
**separate, later, held-out slice of TRAIN** (never seen during fitting or early stopping, and
never the test set).

Artifacts saved to `../models/*.pkl` (gitignored) + `../data/test_predictions.parquet` (gitignored,
consumed by `04_model_evaluation.ipynb`).

In [ ]:
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import TimeSeriesSplit

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 200)

DATA_DIR = Path('../data')
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

## Step 1 — Load engineered frame + tagging dict

In [ ]:
df = pd.read_parquet(DATA_DIR / 'features.parquet')
with open(DATA_DIR / 'feature_tags.json', encoding='utf-8') as f:
    TAGS = json.load(f)

EXP_A_COLS = TAGS['experiment_a_cols']
EXP_B_COLS = TAGS['experiment_b_cols']

print(f"Frame shape: {df.shape}")
print(f"Experiment A columns ({len(EXP_A_COLS)}): {EXP_A_COLS}")
print(f"Experiment B columns ({len(EXP_B_COLS)}): {EXP_B_COLS}")
assert 'elo_diff' not in EXP_A_COLS, "Experiment A must be ELO-free!"
assert 'elo_diff' in EXP_B_COLS, "Experiment B must include elo_diff!"

# liga_tip is a 2-value string category (SM/SF) -- encode as 0/1 so both XGBoost and
# CalibratedClassifierCV get a plain numeric matrix (no categorical dtype plumbing needed
# for a single binary column).
df['liga_tip'] = df['liga_tip'].map({'SM': 0, 'SF': 1}).astype(int)

df = df.sort_values(['Datum', 'SusretId'], kind='mergesort').reset_index(drop=True)
df['Split'].value_counts()

## Step 2 — Train/test split (existing marker from P2 — NOT re-split)

`Split` was assigned in `02_feature_engineering.ipynb` on the 80/20 chronological match boundary
(`split_date` in `feature_tags.json`). Both perspective rows (`winner_pov`/`loser_pov`) of the same
match share the same split by construction.

In [ ]:
train_df = df[df['Split'] == 'train'].copy()
test_df = df[df['Split'] == 'test'].copy()

y_train = train_df['Label'].values
y_test = test_df['Label'].values

print(f"Train: {len(train_df):,} rows  ({train_df['SusretId'].nunique():,} matches)  "
      f"{train_df['Datum'].min().date()} .. {train_df['Datum'].max().date()}")
print(f"Test:  {len(test_df):,} rows  ({test_df['SusretId'].nunique():,} matches)  "
      f"{test_df['Datum'].min().date()} .. {test_df['Datum'].max().date()}")
assert TAGS['n_matches_train'] == train_df['SusretId'].nunique()
assert TAGS['n_matches_test'] == test_df['SusretId'].nunique()
print("OK -- matches feature_tags.json split metadata exactly.")

## Step 3 — Match-level chronological ordering (train only)

All CV / early-stopping / calibration carve-outs below operate on **match** boundaries, not row
boundaries — otherwise a fold split could fall between the `winner_pov` and `loser_pov` rows of the
same match (both rows share the same `Datum`/`SusretId` and are adjacent after the stable sort
above), which would leak the outcome of one row into the other side's "unseen" fold.

In [ ]:
train_match_order = (
    train_df[['SusretId', 'Datum']]
    .drop_duplicates()
    .sort_values(['Datum', 'SusretId'], kind='mergesort')
    .reset_index(drop=True)
)
n_train_matches = len(train_match_order)
assert len(train_df) == 2 * n_train_matches
print(f"Unique train matches: {n_train_matches:,}")

def matches_to_mask(frame, susret_ids):
    return frame['SusretId'].isin(set(susret_ids))

## Step 4 — Hyperparameter sanity check (match-level `TimeSeriesSplit`, TRAIN only)

Small grid over moderate tree depth (4–6) with `reg_alpha`/`reg_lambda` regularization, fixed
`random_state`. Not an exhaustive search — this is a sanity check, not a Kaggle-grade tuning pass.
XGBoost is given the raw NaNs directly (`missing=np.nan` default) — no imputation needed, this is
exactly why XGBoost was chosen (`AgeDiff` alone is ~14.8% NaN).

In [ ]:
N_CV_SPLITS = 4
N_ESTIMATORS_CV = 200

PARAM_GRID = [
    {'max_depth': 4, 'reg_alpha': 0.1, 'reg_lambda': 1.0, 'learning_rate': 0.05},
    {'max_depth': 5, 'reg_alpha': 0.3, 'reg_lambda': 1.5, 'learning_rate': 0.05},
    {'max_depth': 6, 'reg_alpha': 0.5, 'reg_lambda': 2.0, 'learning_rate': 0.05},
]

tscv = TimeSeriesSplit(n_splits=N_CV_SPLITS)
match_idx = np.arange(n_train_matches)

def run_cv(cols, label):
    results = []
    for params in PARAM_GRID:
        fold_briers, fold_loglosses = [], []
        for tr_idx, va_idx in tscv.split(match_idx):
            tr_ids = train_match_order.iloc[tr_idx]['SusretId']
            va_ids = train_match_order.iloc[va_idx]['SusretId']
            tr_mask = matches_to_mask(train_df, tr_ids)
            va_mask = matches_to_mask(train_df, va_ids)
            X_tr, y_tr = train_df.loc[tr_mask, cols], train_df.loc[tr_mask, 'Label']
            X_va, y_va = train_df.loc[va_mask, cols], train_df.loc[va_mask, 'Label']

            model = xgb.XGBClassifier(
                n_estimators=N_ESTIMATORS_CV,
                max_depth=params['max_depth'],
                learning_rate=params['learning_rate'],
                reg_alpha=params['reg_alpha'],
                reg_lambda=params['reg_lambda'],
                subsample=0.9,
                colsample_bytree=0.9,
                random_state=RANDOM_STATE,
                tree_method='hist',
                eval_metric='logloss',
                n_jobs=-1,
            )
            model.fit(X_tr, y_tr)
            p_va = model.predict_proba(X_va)[:, 1]
            fold_briers.append(brier_score_loss(y_va, p_va))
            fold_loglosses.append(log_loss(y_va, np.clip(p_va, 1e-6, 1 - 1e-6)))
        results.append({**params, 'mean_brier': np.mean(fold_briers), 'mean_logloss': np.mean(fold_loglosses)})
    res_df = pd.DataFrame(results).sort_values('mean_brier').reset_index(drop=True)
    print(f"--- CV hyperparameter sanity check: {label} ---")
    print(res_df[['max_depth', 'reg_alpha', 'reg_lambda', 'mean_brier', 'mean_logloss']].to_string(index=False))
    return res_df

cv_results_a = run_cv(EXP_A_COLS, 'Experiment A (ELO-free)')
best_params_a = cv_results_a.iloc[0][['max_depth', 'reg_alpha', 'reg_lambda', 'learning_rate']].to_dict()
best_params_a['max_depth'] = int(best_params_a['max_depth'])
print(f"\nBest params (Exp A): {best_params_a}")

In [ ]:
cv_results_b = run_cv(EXP_B_COLS, 'Experiment B (ELO-included)')
best_params_b = cv_results_b.iloc[0][['max_depth', 'reg_alpha', 'reg_lambda', 'learning_rate']].to_dict()
best_params_b['max_depth'] = int(best_params_b['max_depth'])
print(f"\nBest params (Exp B): {best_params_b}")

## Step 5 — Final fit + calibration (chronological carve within TRAIN)

Within TRAIN, chronologically: `core fit (76.5%)` → `early-stop eval tail (8.5%)` → `calibration
slice (15%)`. The calibration slice is never used for fitting the trees or for early stopping — it
is held out purely to fit the isotonic (or sigmoid, if too small) calibrator, per
`CalibratedClassifierCV(cv='prefit')`. TEST is never touched until Step 6.

In [ ]:
def fit_calibrated_model(cols, best_params, label, n_estimators_final=500,
                          calib_frac=0.15, early_stop_frac=0.10, isotonic_min_rows=1000):
    n_calib = int(round(n_train_matches * calib_frac))
    fit_order = train_match_order.iloc[: n_train_matches - n_calib].reset_index(drop=True)
    calib_match_ids = train_match_order.iloc[n_train_matches - n_calib:]['SusretId']

    n_fit_matches = len(fit_order)
    n_eval = int(round(n_fit_matches * early_stop_frac))
    core_match_ids = fit_order.iloc[: n_fit_matches - n_eval]['SusretId']
    eval_match_ids = fit_order.iloc[n_fit_matches - n_eval:]['SusretId']

    core_mask = matches_to_mask(train_df, core_match_ids)
    eval_mask = matches_to_mask(train_df, eval_match_ids)
    calib_mask = matches_to_mask(train_df, calib_match_ids)

    X_core, y_core = train_df.loc[core_mask, cols], train_df.loc[core_mask, 'Label']
    X_eval, y_eval = train_df.loc[eval_mask, cols], train_df.loc[eval_mask, 'Label']
    X_calib, y_calib = train_df.loc[calib_mask, cols], train_df.loc[calib_mask, 'Label']

    print(f"[{label}] core fit: {len(X_core):,} rows | early-stop eval: {len(X_eval):,} rows | "
          f"calibration: {len(X_calib):,} rows")

    base_model = xgb.XGBClassifier(
        n_estimators=n_estimators_final,
        max_depth=best_params['max_depth'],
        learning_rate=best_params['learning_rate'],
        reg_alpha=best_params['reg_alpha'],
        reg_lambda=best_params['reg_lambda'],
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=RANDOM_STATE,
        tree_method='hist',
        eval_metric='logloss',
        early_stopping_rounds=30,
        n_jobs=-1,
    )
    base_model.fit(X_core, y_core, eval_set=[(X_eval, y_eval)], verbose=False)
    print(f"[{label}] best_iteration = {base_model.best_iteration} / {n_estimators_final}")

    calib_method = 'isotonic' if len(X_calib) >= isotonic_min_rows else 'sigmoid'
    print(f"[{label}] calibration method = {calib_method} (calib rows = {len(X_calib):,}, "
          f"threshold = {isotonic_min_rows:,})")
    calibrated = CalibratedClassifierCV(estimator=base_model, method=calib_method, cv='prefit')
    calibrated.fit(X_calib, y_calib)

    return base_model, calibrated

model_a_raw, model_a_calibrated = fit_calibrated_model(EXP_A_COLS, best_params_a, 'Experiment A')

In [ ]:
model_b_raw, model_b_calibrated = fit_calibrated_model(EXP_B_COLS, best_params_b, 'Experiment B')

## Step 6 — Score the held-out TEST set (Exp 0 / A / B), identical rows for all three

In [ ]:
X_test_a = test_df[EXP_A_COLS]
X_test_b = test_df[EXP_B_COLS]

p_test_a_raw = model_a_raw.predict_proba(X_test_a)[:, 1]
p_test_a_cal = model_a_calibrated.predict_proba(X_test_a)[:, 1]
p_test_b_raw = model_b_raw.predict_proba(X_test_b)[:, 1]
p_test_b_cal = model_b_calibrated.predict_proba(X_test_b)[:, 1]

# Experiment 0: closed-form Elo win-probability formula, no fitting at all
p_test_elo = (1 / (1 + 10 ** (-test_df['elo_diff'] / 400))).values
assert not np.isnan(p_test_elo).any(), "elo_diff should never be NaN (Susret is INNER JOINed on SusretIgracElo)"

def compute_metrics(y_true, p, label):
    p_c = np.clip(p, 1e-6, 1 - 1e-6)
    return {
        'experiment': label,
        'accuracy': accuracy_score(y_true, (p >= 0.5).astype(int)),
        'brier': brier_score_loss(y_true, p),
        'log_loss': log_loss(y_true, p_c),
        'auc': roc_auc_score(y_true, p),
    }

metrics_table = pd.DataFrame([
    compute_metrics(y_test, p_test_elo, 'Exp 0 - ELO baseline'),
    compute_metrics(y_test, p_test_a_cal, 'Exp A - ELO-free XGB (calibrated)'),
    compute_metrics(y_test, p_test_b_cal, 'Exp B - ELO-included XGB (calibrated)'),
])
elo_brier = metrics_table.loc[metrics_table['experiment'] == 'Exp 0 - ELO baseline', 'brier'].values[0]
metrics_table['brier_delta_vs_elo_pct'] = (metrics_table['brier'] - elo_brier) / elo_brier * 100
metrics_table

### Sanity check — calibration effect (raw XGBoost output vs calibrated)

Confirms `CalibratedClassifierCV` is actually doing something (XGBoost's raw sigmoid output
systematically over-concentrates near 0.5 — see plan Part 2.1).

In [ ]:
calibration_check = pd.DataFrame([
    compute_metrics(y_test, p_test_a_raw, 'Exp A - raw (uncalibrated)'),
    compute_metrics(y_test, p_test_a_cal, 'Exp A - calibrated'),
    compute_metrics(y_test, p_test_b_raw, 'Exp B - raw (uncalibrated)'),
    compute_metrics(y_test, p_test_b_cal, 'Exp B - calibrated'),
])
calibration_check

## Step 7 — Save artifacts (gitignored: `../models/*.pkl`, `../data/test_predictions.parquet`)

In [ ]:
with open(MODELS_DIR / 'exp_a_model.pkl', 'wb') as f:
    pickle.dump({
        'raw_model': model_a_raw,
        'calibrated_model': model_a_calibrated,
        'feature_cols': EXP_A_COLS,
        'best_params': best_params_a,
        'cv_results': cv_results_a.to_dict(orient='records'),
    }, f)

with open(MODELS_DIR / 'exp_b_model.pkl', 'wb') as f:
    pickle.dump({
        'raw_model': model_b_raw,
        'calibrated_model': model_b_calibrated,
        'feature_cols': EXP_B_COLS,
        'best_params': best_params_b,
        'cv_results': cv_results_b.to_dict(orient='records'),
    }, f)

test_predictions = test_df[['SusretId', 'Datum', 'SezonaId', 'LigaId', 'PlayerId', 'OpponentId',
                             'Label', 'Perspective', 'elo_diff']].copy()
test_predictions['p_exp0_elo'] = p_test_elo
test_predictions['p_expA_raw'] = p_test_a_raw
test_predictions['p_expA_calibrated'] = p_test_a_cal
test_predictions['p_expB_raw'] = p_test_b_raw
test_predictions['p_expB_calibrated'] = p_test_b_cal
test_predictions.to_parquet(DATA_DIR / 'test_predictions.parquet', index=False)

print(f"Saved: {MODELS_DIR / 'exp_a_model.pkl'}")
print(f"Saved: {MODELS_DIR / 'exp_b_model.pkl'}")
print(f"Saved: {DATA_DIR / 'test_predictions.parquet'}  (shape={test_predictions.shape})")

## Summary

- **Experiment 0** — closed-form ELO formula on the raw `elo_diff` column, no fitting, evaluated
  directly on TEST.
- **Experiment A** — XGBoost on 27 ELO-free columns (`experiment_a_cols`), hyperparameters
  sanity-checked via match-level `TimeSeriesSplit` CV within TRAIN, final model fit on a
  chronological core-fit/early-stop/calibration carve of TRAIN, calibrated via
  `CalibratedClassifierCV(cv='prefit')`.
- **Experiment B** — identical pipeline on 28 columns (Exp A + `elo_diff`).
- All three scored on the exact same 6,852-row TEST set. See `metrics_table` above for the
  headline comparison — full interpretation, calibration curves, and feature importance live in
  `04_model_evaluation.ipynb`.